### **Design**
- **`Config`**: Centralized configuration with properties for directory paths
- **`FontManager`**: Handles font loading and caching
- **`TextProcessor`**: Static methods for text processing operations
- **`ImageGenerator`**: Handles all image creation logic
- **`AudioProcessor`**: Manages TTS pipeline and audio processing
- **`VideoGenerator`**: Main orchestrator class
- **`TextSegment`**: dataclass to represent text with styling info

In [1]:
import os, shutil, subprocess
import re, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any, Tuple
from contextlib import contextmanager

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import textwrap
from PIL import Image, ImageDraw, ImageFont
import soundfile as sf
from kokoro import KPipeline
import torch
import nltk

try:
    nltk.data.find("tokenizers/punkt_tab")
except nltk.downloader.DownloadError:
    nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

In [2]:
# Configuration
@dataclass
class Config:
    screen_size: Tuple[int, int] = (720, 1280)
    font_size: int = 45
    min_font_size: int = 24
    padding: int = 80
    line_spacing: int = 10
    sample_rate: int = 24000
    voice: str = "af_heart"
    speed: float = 1.0

    # Directories
    base_dir: Path = Path("video-resource")

    @property
    def frame_dir(self) -> Path:
        return self.base_dir / "frames"

    @property
    def audio_dir(self) -> Path:
        return self.base_dir / "audio"

    @property
    def output_dir(self) -> Path:
        return self.base_dir / "output"

    @property
    def font_dir(self) -> Path:
        return self.base_dir / "fonts"

    @property
    def fonts(self) -> Dict[str, str]:
        return {
            "bold": str(self.font_dir / "BearSansUI-Bold.otf"),
            "italic": str(self.font_dir / "BearSansUI-Italic.otf"),
            "regular": str(self.font_dir / "BearSansUI-Regular.otf"),
        }

In [3]:
@dataclass
class TextSegment:
    """Represents a segment of text with styling information."""

    text: str
    header: Optional[str] = None
    subtitle: Optional[str] = None
    is_quote: bool = False
    is_summary: bool = False

In [4]:
class FontManager:
    """Manages font loading and caching."""

    def __init__(self, config: Config):
        self.config = config
        self.cache: Dict[str, Dict[int, ImageFont.FreeTypeFont]] = {}
        self._load_fonts()

    def _load_fonts(self):
        """Pre-load and cache fonts in different sizes."""
        for style, path in self.config.fonts.items():
            self.cache[style] = {}
            for size in range(self.config.min_font_size, self.config.font_size + 11):
                try:
                    self.cache[style][size] = ImageFont.truetype(path, size)
                except OSError:
                    self.cache[style][size] = ImageFont.load_default()

    def get_font(self, style: str, size: int) -> ImageFont.FreeTypeFont:
        """Get a font with fallback to default."""
        return self.cache.get(style, {}).get(size, ImageFont.load_default())

In [5]:
class TextProcessor:
    """Handles text processing and chunking operations."""

    @staticmethod
    def split_text(text: str) -> List[str]:
        """Split text into sentences, preserving numbered lists."""
        if not text:
            return []

        numbered_list_pattern = re.compile(r"^\d+\.\s+")
        chunks = []

        for line in text.strip().split("\n"):
            line = line.strip()
            if not line:
                continue
            if numbered_list_pattern.match(line):
                chunks.append(line)
            else:
                chunks.extend(sent_tokenize(line))

        return [s.strip() for s in chunks if s.strip()]

    @staticmethod
    def chunk_long_text(
        text: str, threshold: int = 220, max_length: int = 200
    ) -> List[str]:
        """Break long text into manageable chunks with ellipsis."""
        if len(text) <= threshold:
            return [text]

        chunks = textwrap.wrap(
            text, width=max_length, break_long_words=False, break_on_hyphens=False
        )

        if len(chunks) <= 1:
            return chunks

        modified_chunks = []
        for i, chunk in enumerate(chunks):
            if i > 0:
                chunk = "... " + chunk
            if i < len(chunks) - 1:
                chunk = chunk + "..."
            modified_chunks.append(chunk)

        return modified_chunks

    @staticmethod
    def wrap_text_by_pixels(
        draw: ImageDraw.Draw, text: str, font: ImageFont.FreeTypeFont, max_width: int
    ) -> List[str]:
        """Wrap text to fit within pixel width."""
        lines = []
        words = text.split()
        if not words:
            return []

        current_line = words[0]
        for word in words[1:]:
            test_line = current_line + " " + word
            if draw.textlength(test_line, font=font) <= max_width:
                current_line = test_line
            else:
                lines.append(current_line)
                current_line = word
        lines.append(current_line)
        return lines

In [6]:
class ImageGenerator:
    """Handles image generation for video frames."""

    def __init__(self, config: Config, font_manager: FontManager):
        self.config = config
        self.font_manager = font_manager

    def create_text_image(self, segment: TextSegment) -> Image.Image:
        """Create an image from a text segment."""
        img = Image.new("RGB", self.config.screen_size, "white")
        draw = ImageDraw.Draw(img)

        max_width = self.config.screen_size[0] - 2 * self.config.padding
        y_pos = self.config.padding

        # Draw header
        if segment.header:
            y_pos = self._draw_header(draw, segment.header, max_width, y_pos)

        # Draw subtitle
        if segment.subtitle:
            y_pos = self._draw_subtitle(draw, segment.subtitle, max_width, y_pos)

        # Draw body
        if segment.text:
            self._draw_body(draw, segment, max_width, y_pos)

        return img

    def _draw_header(
        self, draw: ImageDraw.Draw, header: str, max_width: int, y_pos: int
    ) -> int:
        """Draw header text and return new y position."""
        font = self.font_manager.get_font("bold", int(self.config.font_size * 1.2))
        for line in TextProcessor.wrap_text_by_pixels(draw, header, font, max_width):
            draw.text((self.config.padding, y_pos), line, font=font, fill="black")
            y_pos += font.getbbox(line)[3] + self.config.line_spacing
        return y_pos + self.config.line_spacing

    def _draw_subtitle(
        self, draw: ImageDraw.Draw, subtitle: str, max_width: int, y_pos: int
    ) -> int:
        """Draw subtitle text and return new y position."""
        font = self.font_manager.get_font("italic", self.config.font_size)
        for line in TextProcessor.wrap_text_by_pixels(draw, subtitle, font, max_width):
            draw.text((self.config.padding, y_pos), line, font=font, fill="gray")
            y_pos += font.getbbox(line)[3] + self.config.line_spacing
        return y_pos + self.config.line_spacing

    def _draw_body(
        self, draw: ImageDraw.Draw, segment: TextSegment, max_width: int, y_pos: int
    ):
        """Draw body text with appropriate styling."""
        available_height = self.config.screen_size[1] - y_pos - self.config.padding

        # Determine font style and color
        font_style = "italic" if segment.is_quote or segment.is_summary else "regular"
        fill_color = "gray" if segment.is_summary else "black"

        # Find optimal font size
        font_size = self._find_optimal_font_size(
            draw, segment.text, font_style, max_width, available_height
        )
        font = self.font_manager.get_font(font_style, font_size)

        # Wrap text and calculate positioning
        lines = TextProcessor.wrap_text_by_pixels(draw, segment.text, font, max_width)
        total_height = (
            sum(font.getbbox(line)[3] + self.config.line_spacing for line in lines)
            - self.config.line_spacing
        )
        start_y = y_pos + (available_height - total_height) / 2

        # Draw quote/summary bar
        if segment.is_quote or segment.is_summary:
            self._draw_quote_bar(draw, start_y, total_height)

        # Draw text lines
        current_y = start_y
        for line in lines:
            draw.text(
                (self.config.padding, current_y), line, font=font, fill=fill_color
            )
            current_y += font.getbbox(line)[3] + self.config.line_spacing

    def _find_optimal_font_size(
        self,
        draw: ImageDraw.Draw,
        text: str,
        font_style: str,
        max_width: int,
        available_height: int,
    ) -> int:
        """Find the largest font size that fits the available space."""
        for size in range(self.config.font_size, self.config.min_font_size - 1, -2):
            font = self.font_manager.get_font(font_style, size)
            lines = TextProcessor.wrap_text_by_pixels(draw, text, font, max_width)
            total_height = (
                sum(font.getbbox(line)[3] + self.config.line_spacing for line in lines)
                - self.config.line_spacing
            )
            if total_height <= available_height:
                return size
        return self.config.min_font_size

    def _draw_quote_bar(self, draw: ImageDraw.Draw, start_y: float, height: float):
        """Draw a vertical bar for quotes and summaries."""
        bar_width = 4
        bar_padding = 20
        bar_x0 = self.config.padding - bar_padding - bar_width
        bar_x1 = self.config.padding - bar_padding
        draw.rectangle(
            [(bar_x0, start_y), (bar_x1, start_y + height)], fill="lightgray"
        )

In [7]:
class AudioProcessor:
    """Handles audio generation and processing."""

    def __init__(self, config: Config):
        self.config = config
        os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
        self.pipeline = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")

    def generate_audio(self, text: str) -> Tuple[np.ndarray, str]:
        """Generate audio from text and return audio data with synthesized text."""
        full_audio = []
        text_parts = []

        for gs, _, audio in self.pipeline(
            text, voice=self.config.voice, speed=self.config.speed
        ):
            if audio is not None:
                full_audio.append(audio)
                text_parts.append(gs)

        if not full_audio:
            raise ValueError(f"No audio generated for text: '{text}'")

        combined_audio = torch.cat(full_audio).squeeze().cpu().numpy()
        synthesized_text = "".join(text_parts)

        return combined_audio, synthesized_text

    def split_audio_by_chunks(
        self, audio_data: np.ndarray, synthesized_text: str, chunks: List[str]
    ) -> List[np.ndarray]:
        """Split audio data according to text chunks."""
        if len(chunks) <= 1:
            return [audio_data]

        total_samples = len(audio_data)
        total_chars = len(synthesized_text)
        audio_segments = []
        current_sample = 0
        start_search_index = 0

        for chunk in chunks:
            clean_chunk = chunk.replace("...", "").strip()

            try:
                chunk_start = synthesized_text.index(clean_chunk, start_search_index)
                chunk_end = chunk_start + len(clean_chunk)
                start_search_index = chunk_end
            except ValueError:
                chunk_start = 0
                chunk_end = len(clean_chunk)
                total_chars = len(synthesized_text) if total_chars == 0 else total_chars

            chunk_ratio = (chunk_end - chunk_start) / total_chars
            num_samples = int(total_samples * chunk_ratio)

            segment = audio_data[current_sample : current_sample + num_samples]
            audio_segments.append(segment)
            current_sample += num_samples

        return audio_segments

In [8]:
class VideoGenerator:
    """Main class that orchestrates the video generation process."""

    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.font_manager = FontManager(self.config)
        self.image_generator = ImageGenerator(self.config, self.font_manager)
        self.audio_processor = AudioProcessor(self.config)
        self.frame_index = 0

        self._setup_directories()

    def _setup_directories(self):
        """Create necessary directories."""
        for directory in [
            self.config.frame_dir,
            self.config.audio_dir,
            self.config.output_dir,
        ]:
            directory.mkdir(parents=True, exist_ok=True)

    @contextmanager
    def _cleanup_on_error(self):
        """Context manager to cleanup files on error."""
        try:
            yield
        except Exception as e:
            self._cleanup_temp_files()
            raise e

    def _cleanup_temp_files(self):
        """Remove temporary files."""
        for directory in [self.config.frame_dir, self.config.audio_dir]:
            if directory.exists():
                shutil.rmtree(directory)
                directory.mkdir(parents=True, exist_ok=True)

    def process_article(self, json_path: Path) -> str:
        """Process a JSON article and generate video."""
        with open(json_path, "r", encoding="utf-8") as f:
            article = json.load(f)

        title = article.get("title", "")
        subtitle = article.get("subtitle", "")
        sections = article.get("sections", [])

        with self._cleanup_on_error():
            # Process title and subtitle
            self._process_title_and_subtitle(title, subtitle)

            # Process sections
            for section in sections:
                self._process_section(section, title)

            # Generate final video
            output_path = self._render_video(title)
            self._cleanup_temp_files()

            return str(output_path)

    def _process_title_and_subtitle(self, title: str, subtitle: str):
        """Process article title and subtitle."""
        for sentence in TextProcessor.split_text(title):
            self._process_segment(TextSegment(text=sentence, header=sentence))

        for sentence in TextProcessor.split_text(subtitle):
            self._process_segment(
                TextSegment(text=sentence, header=title, subtitle=sentence)
            )

    def _process_section(self, section: Dict[str, Any], main_title: str):
        """Process a single section of the article."""
        section_title = section.get("title", "")
        summary_text = section.get("summary")
        header = (
            section_title if section_title.lower() != main_title.lower() else main_title
        )

        # Process section title
        if section_title and section_title.lower() != main_title.lower():
            for sentence in TextProcessor.split_text(section_title):
                self._process_segment(TextSegment(text=sentence, header=section_title))

        # Process summary
        if summary_text:
            self._process_segment(
                TextSegment(
                    text=f"Here is the summary of {section_title}:", header=header
                )
            )
            for sentence in TextProcessor.split_text(summary_text):
                self._process_segment(
                    TextSegment(text=sentence, header=header, is_summary=True)
                )
            self._process_segment(
                TextSegment(
                    text="End of Summary. Now reading the main article:", header=header
                )
            )

        # Process content
        for para in section.get("content", []):
            is_quote = para.startswith("<start quote>")
            content = (
                para.replace("<start quote>", "").replace("<end quote>", "").strip()
                if is_quote
                else para
            )

            for sentence in TextProcessor.split_text(content):
                self._process_segment(
                    TextSegment(text=sentence, header=header, is_quote=is_quote)
                )

    def _process_segment(self, segment: TextSegment):
        """Process a single text segment into audio and image."""
        chunks = TextProcessor.chunk_long_text(segment.text)

        # Prepare text for speech
        speech_text = segment.text
        if segment.is_quote:
            speech_text = f"Start quote. {segment.text} End quote."

        try:
            audio_data, synthesized_text = self.audio_processor.generate_audio(
                speech_text
            )
        except ValueError as e:
            print(f"Warning: {e}")
            return

        if len(chunks) <= 1:
            # Single chunk - simple processing
            self._save_frame_and_audio(segment, audio_data)
        else:
            # Multiple chunks - split audio
            audio_segments = self.audio_processor.split_audio_by_chunks(
                audio_data, synthesized_text, chunks
            )

            for i, (chunk, audio_segment) in enumerate(zip(chunks, audio_segments)):
                chunk_segment = TextSegment(
                    text=chunk,
                    header=segment.header,
                    subtitle=segment.subtitle,
                    is_quote=segment.is_quote,
                    is_summary=segment.is_summary,
                )
                self._save_frame_and_audio(chunk_segment, audio_segment)

    def _save_frame_and_audio(self, segment: TextSegment, audio_data: np.ndarray):
        """Save image frame and audio file."""
        # Clean display text
        display_text = segment.text
        if segment.is_quote:
            display_text = (
                display_text.replace("Start quote.", "")
                .replace("End quote.", "")
                .strip()
            )

        # Skip redundant display text
        if not segment.is_quote and not segment.is_summary:
            if display_text == segment.header or display_text == segment.subtitle:
                display_text = None

        display_segment = TextSegment(
            text=display_text,
            header=segment.header,
            subtitle=segment.subtitle,
            is_quote=segment.is_quote,
            is_summary=segment.is_summary,
        )

        # Generate and save image
        image = self.image_generator.create_text_image(display_segment)
        image_path = self.config.frame_dir / f"frame_{self.frame_index:04d}.png"
        image.save(image_path)

        # Save audio
        audio_path = self.config.audio_dir / f"part_{self.frame_index:04d}.wav"
        sf.write(str(audio_path), audio_data, self.config.sample_rate)

        self.frame_index += 1

    def _render_video(self, title: str) -> Path:
        """Render final video using FFmpeg."""
        # Create filelist for FFmpeg
        filelist_path = self.config.frame_dir / "filelist.txt"
        with open(filelist_path, "w") as f:
            audio_files = sorted(self.config.audio_dir.glob("part_*.wav"))
            for audio_file in audio_files:
                idx = audio_file.stem.split("_")[1]
                image_file = self.config.frame_dir / f"frame_{idx}.png"

                if image_file.exists():
                    try:
                        duration = sf.info(str(audio_file)).duration
                        if duration >= 0.01:  # Skip very short audio clips
                            f.write(f"file '{image_file.resolve()}'\n")
                            f.write(f"duration {duration}\n")
                    except Exception:
                        continue

        # Combine all audio
        master_audio_path = self.config.audio_dir / "master_audio.wav"
        self._combine_audio_files(master_audio_path)

        # Run FFmpeg
        output_path = self.config.output_dir / f"{title}.mp4"
        command = [
            "ffmpeg",
            "-f",
            "concat",
            "-safe",
            "0",
            "-i",
            str(filelist_path),
            "-i",
            str(master_audio_path),
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-c:a",
            "copy",
            "-shortest",
            "-y",
            str(output_path),
        ]

        try:
            subprocess.run(command, check=True, capture_output=True, text=True)
            print(f"✅ Video successfully generated: {output_path}")
            return output_path
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"FFmpeg rendering failed: {e.stderr}")
        except FileNotFoundError:
            raise RuntimeError("FFmpeg not found. Please install FFmpeg.")

    def _combine_audio_files(self, output_path: Path):
        """Combine all audio files into a master audio file."""
        audio_files = sorted(self.config.audio_dir.glob("part_*.wav"))
        if not audio_files:
            raise ValueError("No audio files found to combine")

        combined_data = []
        for file_path in audio_files:
            data, sample_rate = sf.read(file_path)
            if sample_rate == self.config.sample_rate:
                combined_data.append(data)

        if combined_data:
            master_audio = np.concatenate(combined_data)
            sf.write(output_path, master_audio, self.config.sample_rate)

In [ ]:
# Example usage
config = Config()
generator = VideoGenerator(config)

# Process article
json_path = Path(
    "video-resource/json-input/Money Stuff - A Drug-Trial Stock Sale.json"
)  # Replace with your JSON file path
output_video = generator.process_article(json_path)

/Users/rehabnaeem/Developer/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/Users/rehabnaeem/Developer/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ Video successfully generated: video-resource/output/A Drug-Trial Stock Sale.mp4
Video generated successfully: video-resource/output/A Drug-Trial Stock Sale.mp4


Updates:
- Consider adding images from main article
- Incorporate "multiprocessing" to accelerate processing